In [65]:
import pandas as pd
import altair as alt

# ==========================================
# CONFIGURACIÓN: CAMBIA EL AÑO AQUÍ
# ==========================================
anio_seleccionado = 2025
# ==========================================

# 1. Cargar el archivo
df = pd.read_csv('movimientos_y_tabla.csv', sep=';', encoding='latin-1')

# 2. Limpieza de datos
df['Equipo'] = (df['Equipo']
                .str.replace('ó', 'o')
                .str.replace('ñ', 'n')
                .str.replace('Catolica', 'Católica'))

# Convertir a números
df['Posicion'] = pd.to_numeric(df['Posicion'], errors='coerce')
df['Total altas'] = pd.to_numeric(df['Total altas'], errors='coerce')

# 3. Filtrar por el año configurado
df_year = df[df['Temporada'] == anio_seleccionado].copy()

# 4. Crear el Gráfico de Tabla de Posiciones con Datos de Altas
tabla = alt.Chart(df_year).mark_bar(cornerRadiusEnd=4).encode(
    y=alt.Y('Posicion:O',
            title='Posición en la Tabla',
            sort='ascending'), # El 1 arriba
    x=alt.X('Total altas:Q',
            title='Cantidad de Fichajes (Altas)'),
    color=alt.Color('Posicion:Q',
                    scale=alt.Scale(scheme='magma', reverse=True),
                    legend=None),
    tooltip=[
        alt.Tooltip('Posicion:Q', title='Lugar en Tabla'),
        alt.Tooltip('Equipo:N', title='Club'),
        alt.Tooltip('Total altas:Q', title='Fichajes Totales'),
        alt.Tooltip('Altas por traspaso:Q', title='Por Traspaso'),
        alt.Tooltip('Altas libre:Q', title='Libres')
    ]
).properties(
    title=f'Tabla de Posiciones vs. Fichajes - Temporada {anio_seleccionado}',
    width=600,
    height=500
)

# 5. Añadir etiquetas de texto (Nombre del equipo + Cantidad de altas)
etiquetas = tabla.mark_text(
    align='left',
    baseline='middle',
    dx=5,
    fontWeight='bold'
).encode(
    text=alt.Text('etiqueta:N')
).transform_calculate(
    # Creamos una etiqueta combinada: "Equipo (N° altas)"
    etiqueta = "datum.Equipo + ' (' + datum['Total altas'] + ' altas)'"
)

# Visualizar
(tabla + etiquetas).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=18,
    anchor='start'
)

alt.LayerChart(...)